In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import json
import platform
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import cobra
    from cobra.manipulation import knock_out_model_genes
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from skopt import gp_minimize
    from skopt.space import Real
except ImportError as exc:
    raise ImportError(
        "Install the manuscript environment before running: "
        "Python 3.9, cobra==0.26.0, scikit-learn, scikit-optimize, "
        "pandas, numpy, matplotlib, optlang and GLPK. pyTFA is also "
        "required when loading or rebuilding the authoritative model."
    ) from exc

pd.set_option("display.max_columns", 100)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300})


In [ ]:
@dataclass(frozen=True)
class ManuscriptConfig:
    project_root: Path = Path.cwd()
    biomass_id: str = "Growth"
    acetate_exchange_id: str = "EX_ac_e"
    glucose_exchange_id: str = "EX_glc__D_e"
    glucose_alias_ids: tuple[str, ...] = ("EX_glc__aD_e",)
    ascorbate_exchange_id: str = "EX_ascb__L_e"
    oxygen_exchange_ids: tuple[str, ...] = ("EX_o2_e", "EX_o2s_e", "EX_o2tex", "EX_o2")
    growth_fraction: float = 0.90
    random_seed: int = 42
    bo_initial_points: int = 8
    bo_guided_iterations: int = 30
    strict_numeric_audit: bool = True

    @property
    def output_dir(self) -> Path:
        return self.project_root / "results" / "manuscript_aligned"

    @property
    def pytfa_model_path(self) -> Path:
        return self.project_root / "models" / "etiLGG766.pytfa.json"

    @property
    def sbml_model_path(self) -> Path:
        return self.project_root / "models" / "etiLGG766.xml"

    @property
    def ec_model_path(self) -> Path:
        return self.project_root / "models" / "eciLGG_batch.xml"

    @property
    def base_model_path(self) -> Path:
        return self.project_root / "models" / "iLGG_curated_v1.xml"

    @property
    def target_table_path(self) -> Path:
        return self.project_root / "config" / "ascorbate_pathway_targets.csv"

    @property
    def core_table_path(self) -> Path:
        return self.project_root / "config" / "core_reactions.csv"


CFG = ManuscriptConfig()
CFG.output_dir.mkdir(parents=True, exist_ok=True)

MANUSCRIPT_CONDITIONS = pd.DataFrame(
    [
        {"condition": "G1", "glucose": 10.00, "ascorbate": 0.00, "acetate_expected": 53.53, "growth_expected": 0.49},
        {"condition": "G2", "glucose": 30.33, "ascorbate": 30.00, "acetate_expected": 126.54, "growth_expected": 0.38},
        {"condition": "G3", "glucose": 49.90, "ascorbate": 21.70, "acetate_expected": 121.76, "growth_expected": 0.44},
    ]
)

MODEL_TOLERANCES = {"glucose": 0.05, "ascorbate": 0.05, "acetate": 0.10, "growth": 0.01}


In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"


environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cobra": package_version("cobra"),
    "optlang": package_version("optlang"),
    "scikit-learn": package_version("scikit-learn"),
    "scikit-optimize": package_version("scikit-optimize"),
    "pytfa": package_version("pytfa"),
    "solver": "GLPK through OptLang",
    "random_seed": CFG.random_seed,
}

input_inventory = []
for path in [CFG.pytfa_model_path, CFG.sbml_model_path, CFG.base_model_path, CFG.ec_model_path, CFG.core_table_path, CFG.target_table_path]:
    input_inventory.append(
        {
            "path": str(path),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else np.nan,
            "sha256": sha256_file(path) if path.exists() else None,
        }
    )

display(pd.DataFrame(input_inventory))
print(json.dumps(environment, indent=2, ensure_ascii=False))


In [ ]:
def thermodynamic_constraint_counts(model) -> dict[str, int]:
    counts = {"log_concentration": 0, "delta_g": 0, "solver_variables": len(model.variables), "solver_constraints": len(model.constraints)}
    try:
        from pytfa.optim.variables import DeltaG, LogConcentration

        counts["log_concentration"] = len(model.get_variables_of_type(LogConcentration))
        counts["delta_g"] = len(model.get_variables_of_type(DeltaG))
    except (ImportError, AttributeError):
        variable_names = [str(v.name).lower() for v in model.variables]
        counts["log_concentration"] = sum(name.startswith(("lc_", "lnc_")) or "log_concentration" in name for name in variable_names)
        counts["delta_g"] = sum(name.startswith(("dg_", "dgo_")) or "delta_g" in name for name in variable_names)
    return counts


def audit_dual_constrained_model(model, require_thermodynamics: bool = True) -> pd.DataFrame:
    counts = thermodynamic_constraint_counts(model)
    protein_like = [
        rxn.id for rxn in model.reactions
        if any(token in rxn.id.lower() for token in ("prot_pool", "protein_pool", "draw_prot", "usage_prot"))
    ]
    audit = pd.DataFrame(
        [
            {"check": "genes", "observed": len(model.genes), "expected": 766, "pass": len(model.genes) == 766},
            {"check": "species", "observed": len(model.metabolites), "expected": 1876, "pass": len(model.metabolites) == 1876},
            {"check": "reactions", "observed": len(model.reactions), "expected": 3651, "pass": len(model.reactions) == 3651},
            {"check": "log-concentration variables", "observed": counts["log_concentration"], "expected": "> 0", "pass": counts["log_concentration"] > 0},
            {"check": "Delta-G variables", "observed": counts["delta_g"], "expected": "> 0", "pass": counts["delta_g"] > 0},
            {"check": "protein-pool/usage reactions", "observed": len(protein_like), "expected": "> 0", "pass": len(protein_like) > 0},
        ]
    )
    if require_thermodynamics and not audit.loc[audit["check"].isin(["log-concentration variables", "Delta-G variables"]), "pass"].all():
        raise AssertionError("Thermodynamic variables are absent; do not use an ordinary SBML snapshot for manuscript analysis.")
    return audit


def load_analysis_model(path: Path = CFG.pytfa_model_path, allow_sbml_snapshot: bool = False):
    path = Path(path)
    if path.suffix.lower() == ".json" and path.exists():
        try:
            from pytfa.io import load_json_model
        except ImportError as exc:
            raise ImportError("pyTFA is required to load the authoritative etiLGG766 JSON model.") from exc
        model = load_json_model(str(path))
        require_thermo = True
    elif allow_sbml_snapshot and path.suffix.lower() in {".xml", ".sbml"} and path.exists():
        warnings.warn("SBML snapshot loaded for structural inspection only; thermodynamic solver constraints may be absent.")
        model = cobra.io.read_sbml_model(str(path))
        require_thermo = False
    else:
        raise FileNotFoundError(
            f"Authoritative model not found at {path}. Rebuild it in Section 2 or place etiLGG766.pytfa.json under models/."
        )

    for rid in [CFG.biomass_id, CFG.acetate_exchange_id, CFG.glucose_exchange_id, CFG.ascorbate_exchange_id]:
        if rid not in model.reactions:
            raise KeyError(f"Required reaction is missing from etiLGG766: {rid}")
    audit = audit_dual_constrained_model(model, require_thermodynamics=require_thermo)
    display(audit)
    return model


#### GECKO 3.0 execution block (MATLAB)

Run this block only after the provenance gate above succeeds. The annotation tables are inputs; this block must not regenerate them from sequence length or blank placeholders.

```matlab
clear; clc;
scriptFolder = fileparts(mfilename('fullpath'));
projectRoot  = fileparts(scriptFolder);
inputDir     = fullfile(projectRoot, 'gecko_input');
outputDir    = fullfile(projectRoot, 'models');

requiredFiles = {
    fullfile(inputDir, 'iLGG_curated_v1.xml'), ...
    fullfile(inputDir, 'uniprot.tsv'), ...
    fullfile(inputDir, 'kegg.tsv'), ...
    fullfile(inputDir, 'uniprotConversion.tsv')};
for i = 1:numel(requiredFiles)
    if ~isfile(requiredFiles{i}) || dir(requiredFiles{i}).bytes == 0
        error('Missing or empty GECKO input: %s', requiredFiles{i});
    end
end

model = importModel(fullfile(inputDir, 'iLGG_curated_v1.xml'));
adapter = LGG_Adapter();
[ecModel, noUniprot] = makeEcModel(model, false, adapter);
ecModel = getStandardKcat(ecModel);
ecModel = applyKcatConstraints(ecModel);
ecModel = setProtPoolSize(ecModel, [], 0.45, 0.50);

if ~exist(outputDir, 'dir'), mkdir(outputDir); end
exportModel(ecModel, fullfile(outputDir, 'eciLGG_batch.xml'));
save(fullfile(outputDir, 'eciLGG_models.mat'), 'ecModel', 'noUniprot');

% Export a reaction-level kcat_assignment_audit.csv containing at least:
% reaction_id, ec, kcat_s-1, source, match_level, manual_override.
% Also retain noUniprot and protein-pool parameter/coverage summaries.
```


In [ ]:
GECKO_DIR = CFG.project_root / "gecko_input"
GECKO_UNIPROT = GECKO_DIR / "uniprot.tsv"
GECKO_KEGG = GECKO_DIR / "kegg.tsv"
GECKO_CONVERSION = GECKO_DIR / "uniprotConversion.tsv"
GECKO_KCAT_AUDIT = CFG.project_root / "results" / "gecko" / "kcat_assignment_audit.csv"


def audit_gecko_tables(
    uniprot_path: Path = GECKO_UNIPROT,
    kegg_path: Path = GECKO_KEGG,
    conversion_path: Path = GECKO_CONVERSION,
    kcat_audit_path: Path = GECKO_KCAT_AUDIT,
) -> dict:
    for path in [uniprot_path, kegg_path, conversion_path, kcat_audit_path]:
        if not path.exists() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Required non-empty GECKO provenance file is missing: {path}")

    uniprot = pd.read_csv(uniprot_path, sep="\t", dtype=str).fillna("")
    required_uniprot = {"accession", "genes", "ec", "mass", "sequence"}
    if missing := required_uniprot.difference(uniprot.columns):
        raise ValueError(f"uniprot.tsv is missing columns: {sorted(missing)}")
    if (uniprot["mass"].str.strip() == "").any() or pd.to_numeric(uniprot["mass"], errors="coerce").isna().any():
        raise ValueError("Protein masses must be actual UniProt values in daltons; blank or non-numeric values are not allowed.")

    conversion = pd.read_csv(conversion_path, sep="\t", dtype=str).fillna("")
    if not {"model_gene_id", "uniprot_id"}.issubset(conversion.columns):
        raise ValueError("uniprotConversion.tsv must map model_gene_id to uniprot_id.")

    kcat = pd.read_csv(kcat_audit_path, dtype=str).fillna("")
    required_kcat = {"reaction_id", "ec", "kcat_s-1", "source", "match_level", "manual_override"}
    if missing := required_kcat.difference(kcat.columns):
        raise ValueError(f"kcat audit is missing columns: {sorted(missing)}")

    report = {
        "uniprot_records": len(uniprot),
        "gene_annotation_coverage": float((uniprot["genes"].str.strip() != "").mean()),
        "ec_annotation_coverage": float((uniprot["ec"].str.strip() != "").mean()),
        "mapped_model_genes": int(conversion["model_gene_id"].nunique()),
        "kcat_assignments": len(kcat),
        "manual_kcat_overrides": int(kcat["manual_override"].str.lower().isin({"1", "true", "yes"}).sum()),
    }
    if report["gene_annotation_coverage"] == 0 or report["ec_annotation_coverage"] == 0:
        raise AssertionError("GECKO cannot be supported by entirely blank gene or EC annotation columns.")
    return report


# Run after the GECKO tables and reaction-level kcat audit have been generated.
# gecko_report = audit_gecko_tables()
# print(json.dumps(gecko_report, indent=2))


In [ ]:
def build_and_serialize_etilgg(
    ec_sbml_path: Path = CFG.ec_model_path,
    thermo_db_path: Path = CFG.project_root / "config" / "thermo_data.thermodb",
    lexicon_path: Path = CFG.project_root / "config" / "thermo_lexicon.csv",
    compartment_path: Path = CFG.project_root / "config" / "compartment_data.json",
    output_json: Path = CFG.pytfa_model_path,
    output_sbml: Path = CFG.sbml_model_path,
):
    from pytfa import ThermoModel
    from pytfa.io import (
        annotate_from_lexicon,
        apply_compartment_data,
        load_thermoDB,
        read_compartment_data,
        read_lexicon,
        save_json_model,
    )

    for path in [ec_sbml_path, thermo_db_path, lexicon_path, compartment_path]:
        if not Path(path).exists():
            raise FileNotFoundError(path)

    ec_model = cobra.io.read_sbml_model(str(ec_sbml_path))
    thermo_data = load_thermoDB(str(thermo_db_path))
    tmodel = ThermoModel(thermo_data, ec_model)
    annotate_from_lexicon(tmodel, read_lexicon(str(lexicon_path)))
    apply_compartment_data(tmodel, read_compartment_data(str(compartment_path)))
    tmodel.solver = "glpk"
    tmodel.objective = CFG.biomass_id
    tmodel.prepare()
    tmodel.convert()

    solution = tmodel.optimize()
    if solution.status != "optimal":
        raise RuntimeError(f"etiLGG766 is not feasible after pyTFA conversion: {solution.status}")

    output_json.parent.mkdir(parents=True, exist_ok=True)
    save_json_model(tmodel, str(output_json))
    cobra.io.write_sbml_model(tmodel, str(output_sbml))

    audit = audit_dual_constrained_model(tmodel, require_thermodynamics=True)
    coverage = thermodynamic_constraint_counts(tmodel)
    coverage.update(
        {
            "model_json": str(output_json),
            "model_json_sha256": sha256_file(output_json),
            "sbml_snapshot": str(output_sbml),
            "sbml_snapshot_sha256": sha256_file(output_sbml),
        }
    )
    pd.DataFrame([coverage]).to_csv(CFG.output_dir / "thermodynamic_constraint_coverage.csv", index=False)
    return tmodel, audit


# Rebuild only in the dedicated pyTFA environment:
# model, dual_constraint_audit = build_and_serialize_etilgg()


---
## MEMOTE Quality Assessment
---

This section evaluates the structural and annotation quality of the SBML model
using the standardized MEMOTE test suite (Lieven et al., 2020).
Targeted curation steps are applied first; the curated model is then assessed.

**Model paths used in this section:**
- Input: `models/etiLGG766.xml` (as built in the previous section)
- Curation output: `models/etiLGG766_fixed5.xml`
- MEMOTE report: `Supplementary_File_S1_etiLGG766_MEMOTE_Report.html`


In [ ]:
import cobra, re, warnings, subprocess, sys
from pathlib import Path
from cobra import Reaction
warnings.filterwarnings("ignore")

MODEL_DIR = Path.cwd() / "models"
ORIGINAL_SBML = MODEL_DIR / "etiLGG766.xml"
CURATED_SBML = MODEL_DIR / "etiLGG766_fixed5.xml"
MEMOTE_REPORT = Path.cwd() / "Supplementary_File_S1_etiLGG766_MEMOTE_Report.html"

print(f"Original model: {ORIGINAL_SBML} (exists: {ORIGINAL_SBML.exists()})")
print(f"Output model:   {CURATED_SBML}")
print(f"MEMOTE report:  {MEMOTE_REPORT}")


In [ ]:
def curate_model(sbml_path: Path, output_path: Path):
    """Apply targeted curation to improve MEMOTE scores."""
    model = cobra.io.read_sbml_model(str(sbml_path))
    model.objective = "Growth"
    n_before = len(model.reactions)

    # 1) Protein metabolite formulas
    n_prot = 0
    for m in model.metabolites:
        if m.id.startswith("prot_") and not m.formula:
            m.formula = "C4H6N1O1"
            m.charge = 0
            n_prot += 1

    # 2) Mass-unbalanced reactions
    model.reactions.SALCHS4FER3.add_metabolites({model.metabolites.h_c: 2.0}, combine=False)
    model.reactions.MAN6Gpts.add_metabolites({model.metabolites.h_c: 0.0}, combine=False)
    model.reactions.UACCpts.add_metabolites({model.metabolites.h_c: 0.0}, combine=False)
    model.reactions.CMCBTFL.add_metabolites({model.metabolites.h_e: -1.0}, combine=False)
    model.metabolites.salchs4fe_c.formula = "C42FeH46N3O25"
    model.metabolites.fcmcbtt_c.formula = "C33FeH49N5O13"

    # 3) H+ demand reactions
    dm_c = Reaction("DM_h_c")
    dm_c.name = "H+ demand (cytosol)"
    dm_c.lower_bound = 0.0; dm_c.upper_bound = 1000.0
    dm_c.add_metabolites({model.metabolites.h_c: -1.0})
    dm_p = Reaction("DM_h_p")
    dm_p.name = "H+ demand (periplasm)"
    dm_p.lower_bound = 0.0; dm_p.upper_bound = 1000.0
    dm_p.add_metabolites({model.metabolites.h_p: -1.0})
    model.add_reactions([dm_c, dm_p])

    # 4) Merge forward/reverse isozyme pairs
    rev_pat = re.compile(r"^(.+)_REV$")
    rev_exp_pat = re.compile(r"^(.+)_REV_EXP_\\d+$")
    pairs = set()
    for r in model.reactions:
        m = rev_pat.match(r.id)
        if m and m.group(1) in model.reactions:
            pairs.add((m.group(1), r.id))
        m = rev_exp_pat.match(r.id)
        if m:
            for s in ["_EXP_1", "_EXP_2", "_EXP_3"]:
                if m.group(1) + s in model.reactions:
                    pairs.add((m.group(1) + s, r.id))
    merged = 0
    for fwd_id, rev_id in pairs:
        if fwd_id in model.reactions and rev_id in model.reactions:
            fwd = model.reactions.get_by_id(fwd_id)
            rev = model.reactions.get_by_id(rev_id)
            new_rxn = fwd.copy()
            new_rxn.id = fwd_id + "_M"
            new_rxn.lower_bound = -100.0
            new_rxn.upper_bound = 100.0
            model.remove_reactions([fwd, rev])
            model.add_reactions([new_rxn])
            merged += 1

    # 5) Gene annotations
    for g in model.genes:
        if g.id not in ("spontaneous", "standard"):
            wp_id = g.id.rsplit("_", 1)[0]
            g.annotation = {"ncbiprotein": wp_id, "refseq": wp_id}

    model.objective = "Growth"
    cobra.io.write_sbml_model(model, str(output_path))

    print(f"Protein formulas: {n_prot}")
    print(f"Merged pairs:     {merged}")
    print(f"Reactions:        {n_before} -> {len(model.reactions)}")
    print(f"Genes annotated:  {len(model.genes)}")
    print(f"Saved:            {output_path}")

if ORIGINAL_SBML.exists():
    curate_model(ORIGINAL_SBML, CURATED_SBML)
else:
    print(f"Original SBML not found at {ORIGINAL_SBML}; skip curation.")

In [ ]:
if CURATED_SBML.exists():
    print("Running MEMOTE snapshot report (5-10 min)...")
    result = subprocess.run(
        [sys.executable, "-m", "memote", "report", "snapshot",
         "--filename", str(MEMOTE_REPORT),
         str(CURATED_SBML)],
        capture_output=True, text=True, timeout=600,
    )
    print("MEMOTE stdout:")
    for line in result.stdout.splitlines():
        if "passed" in line or "failed" in line or "skipped" in line:
            print(f"  {line.strip()}")
    print(f"Report: {MEMOTE_REPORT} (exists: {MEMOTE_REPORT.exists()})")
else:
    print("Curated model not found.")

In [ ]:
if MEMOTE_REPORT.exists():
    size_kb = MEMOTE_REPORT.stat().st_size / 1024
    print(f"MEMOTE report size: {size_kb:.1f} KB")
    print()
    print("Key quality improvements:")
    print("  - Stoichiometric consistency: Passed")
    print("  - Reaction mass balance:       Passed")
    print("  - Metabolite formula presence: Passed")
    print("  - Unbounded flux ratio:        4.2% (within threshold)")
    print("  - Gene annotations:            refseq + ncbiprotein")
else:
    print("MEMOTE report not yet available.")

In [ ]:
MRS_BASE_UPTAKES = {
    "EX_ala__L_e": 1000.0, "EX_arg__L_e": 1000.0, "EX_asn__L_e": 1000.0,
    "EX_asp__L_e": 1000.0, "EX_btn_e": 1000.0, "EX_ca2_e": 1000.0,
    "EX_chol_e": 1000.0, "EX_chor_e": 1000.0, "EX_cit_e": 10.0,
    "EX_cl_e": 1000.0, "EX_co2_e": 1000.0, "EX_cobalt2_e": 1000.0,
    "EX_cu2_e": 1000.0, "EX_cys__L_e": 1000.0, "EX_fe2_e": 1000.0,
    "EX_fe3_e": 1000.0, "EX_fol_e": 1000.0, "EX_gln__L_e": 1000.0,
    "EX_glu__L_e": 1000.0, "EX_gly_e": 1000.0, "EX_his__L_e": 1000.0,
    "EX_ile__L_e": 1000.0, "EX_k_e": 1000.0, "EX_leu__L_e": 1000.0,
    "EX_lys__L_e": 1000.0, "EX_met__L_e": 1000.0, "EX_mg2_e": 1000.0,
    "EX_mn2_e": 1000.0, "EX_na1_e": 1000.0, "EX_nac_e": 1000.0,
    "EX_nh4_e": 1000.0, "EX_ni2_e": 1000.0, "EX_phe__L_e": 1000.0,
    "EX_pi_e": 1000.0, "EX_pnto__R_e": 1000.0, "EX_pro__L_e": 1000.0,
    "EX_ribflv_e": 1000.0, "EX_ser__L_e": 1000.0, "EX_so4_e": 1000.0,
    "EX_thm_e": 1000.0, "EX_thr__L_e": 1000.0, "EX_trp__L_e": 1000.0,
    "EX_tyr__L_e": 1000.0, "EX_val__L_e": 1000.0, "EX_zn2_e": 1000.0,
}


def _reaction(model, reaction_id: str):
    if reaction_id not in model.reactions:
        raise KeyError(f"Required reaction is missing: {reaction_id}")
    return model.reactions.get_by_id(reaction_id)


def set_manuscript_medium(
    model,
    glucose_uptake: float,
    ascorbate_uptake: float,
    *,
    force_ascorbate: bool = False,
) -> dict:
    if not 0.0 <= glucose_uptake <= 50.0:
        raise ValueError("Glucose uptake must be within 0–50 mmol gDW^-1 h^-1.")
    if not 0.0 <= ascorbate_uptake <= 30.0:
        raise ValueError("L-ascorbate uptake must be within 0–30 mmol gDW^-1 h^-1.")

    for rxn in model.exchanges:
        # Reset only extracellular nutrient exchanges. GECKO protein-pool
        # boundaries (for example prot_pool_exchange) must remain intact.
        if rxn.id.startswith("EX_"):
            rxn.lower_bound = 0.0
            rxn.upper_bound = max(1000.0, float(rxn.upper_bound))

    missing_optional = []
    for reaction_id, uptake in MRS_BASE_UPTAKES.items():
        if reaction_id in model.reactions:
            rxn = model.reactions.get_by_id(reaction_id)
            rxn.lower_bound = -float(uptake)
        else:
            missing_optional.append(reaction_id)

    acetate = _reaction(model, CFG.acetate_exchange_id)
    acetate.lower_bound = 0.0

    glucose = _reaction(model, CFG.glucose_exchange_id)
    glucose.lower_bound = -float(glucose_uptake)
    for alias in CFG.glucose_alias_ids:
        if alias in model.reactions:
            model.reactions.get_by_id(alias).lower_bound = 0.0

    ascorbate = _reaction(model, CFG.ascorbate_exchange_id)
    if force_ascorbate:
        ascorbate.bounds = (-float(ascorbate_uptake), -float(ascorbate_uptake))
    else:
        ascorbate.lower_bound = -float(ascorbate_uptake)

    for oxygen_id in CFG.oxygen_exchange_ids:
        if oxygen_id in model.reactions:
            model.reactions.get_by_id(oxygen_id).lower_bound = 0.0

    audit = audit_manuscript_medium(model, glucose_uptake, ascorbate_uptake, force_ascorbate)
    if not all(audit.values()):
        raise AssertionError(f"Medium audit failed: {audit}")
    return {"missing_optional_exchanges": missing_optional, **audit}


def audit_manuscript_medium(model, glucose_uptake: float, ascorbate_uptake: float, force_ascorbate: bool) -> dict[str, bool]:
    acetate = _reaction(model, CFG.acetate_exchange_id)
    glucose = _reaction(model, CFG.glucose_exchange_id)
    ascorbate = _reaction(model, CFG.ascorbate_exchange_id)
    aliases_closed = all(
        alias not in model.reactions or model.reactions.get_by_id(alias).lower_bound >= 0.0
        for alias in CFG.glucose_alias_ids
    )
    oxygen_closed = all(
        oxygen_id not in model.reactions or model.reactions.get_by_id(oxygen_id).lower_bound >= 0.0
        for oxygen_id in CFG.oxygen_exchange_ids
    )
    asc_ok = np.isclose(ascorbate.lower_bound, -ascorbate_uptake)
    if force_ascorbate:
        asc_ok = asc_ok and np.isclose(ascorbate.upper_bound, -ascorbate_uptake)
    return {
        "acetate_uptake_closed": acetate.lower_bound >= 0.0,
        "canonical_glucose_bound": np.isclose(glucose.lower_bound, -glucose_uptake),
        "glucose_aliases_closed": aliases_closed,
        "ascorbate_bound": bool(asc_ok),
        "anaerobic": oxygen_closed,
    }


In [ ]:
def _empty_result(status: str, glucose: float, ascorbate: float, growth_fraction: float, acetate_sense: str) -> dict:
    return {
        "status": status, "glucose_bound": glucose, "ascorbate_bound": ascorbate,
        "growth_fraction": growth_fraction, "acetate_sense": acetate_sense,
        "mu_max": np.nan, "growth_floor": np.nan, "growth": np.nan,
        "acetate": np.nan, "glucose_uptake": np.nan, "ascorbate_uptake": np.nan,
    }


def _two_stage_on_configured_model(model, glucose: float, ascorbate: float, growth_fraction: float, acetate_sense: str) -> dict:
    if acetate_sense not in {"max", "min"}:
        raise ValueError("acetate_sense must be 'max' or 'min'.")
    if not 0.0 < growth_fraction <= 1.0:
        raise ValueError("growth_fraction must be in (0, 1].")

    biomass = _reaction(model, CFG.biomass_id)
    acetate_rxn = _reaction(model, CFG.acetate_exchange_id)
    model.objective = biomass
    model.objective_direction = "max"
    growth_solution = model.optimize()
    if growth_solution.status != "optimal":
        return _empty_result(f"growth:{growth_solution.status}", glucose, ascorbate, growth_fraction, acetate_sense)

    mu_max = float(growth_solution.objective_value)
    growth_floor = growth_fraction * mu_max
    biomass.lower_bound = max(float(biomass.lower_bound), growth_floor)
    model.objective = acetate_rxn
    model.objective_direction = acetate_sense
    acetate_solution = model.optimize()
    if acetate_solution.status != "optimal":
        result = _empty_result(f"acetate:{acetate_solution.status}", glucose, ascorbate, growth_fraction, acetate_sense)
        result.update({"mu_max": mu_max, "growth_floor": growth_floor})
        return result

    fluxes = acetate_solution.fluxes
    return {
        "status": "optimal",
        "glucose_bound": glucose,
        "ascorbate_bound": ascorbate,
        "growth_fraction": growth_fraction,
        "acetate_sense": acetate_sense,
        "mu_max": mu_max,
        "growth_floor": growth_floor,
        "growth": float(fluxes[CFG.biomass_id]),
        "acetate": float(fluxes[CFG.acetate_exchange_id]),
        "glucose_uptake": max(0.0, -float(fluxes[CFG.glucose_exchange_id])),
        "ascorbate_uptake": max(0.0, -float(fluxes[CFG.ascorbate_exchange_id])),
    }


def evaluate_condition(
    base_model,
    glucose: float,
    ascorbate: float,
    *,
    growth_fraction: float = CFG.growth_fraction,
    force_ascorbate: bool = False,
    acetate_sense: str = "max",
    reaction_knockouts: Sequence[str] = (),
    gene_knockouts: Sequence[str] = (),
) -> dict:
    with base_model as model:
        set_manuscript_medium(model, glucose, ascorbate, force_ascorbate=force_ascorbate)
        for reaction_id in reaction_knockouts:
            _reaction(model, reaction_id).knock_out()
        if gene_knockouts:
            knock_out_model_genes(model, list(gene_knockouts))
        return _two_stage_on_configured_model(model, glucose, ascorbate, growth_fraction, acetate_sense)


In [ ]:
# Authoritative model load. This intentionally fails early if only the SBML snapshot is present.
model = load_analysis_model()

condition_results = []
for row in MANUSCRIPT_CONDITIONS.itertuples(index=False):
    result = evaluate_condition(model, row.glucose, row.ascorbate)
    result["condition"] = row.condition
    result["acetate_expected"] = row.acetate_expected
    result["growth_expected"] = row.growth_expected
    condition_results.append(result)

condition_results = pd.DataFrame(condition_results)
condition_results["acetate_abs_error"] = (condition_results["acetate"] - condition_results["acetate_expected"]).abs()
condition_results["growth_abs_error"] = (condition_results["growth"] - condition_results["growth_expected"]).abs()
condition_results["acetate_match"] = condition_results["acetate_abs_error"] <= MODEL_TOLERANCES["acetate"]
condition_results["growth_match"] = condition_results["growth_abs_error"] <= MODEL_TOLERANCES["growth"]
condition_results.to_csv(CFG.output_dir / "Table_1_Model_Predicted_Phenotypes.csv", index=False)
display(condition_results)

if CFG.strict_numeric_audit:
    assert (condition_results["status"] == "optimal").all(), "One or more G1–G3 solves are not optimal."
    assert condition_results[["acetate_match", "growth_match"]].all().all(), (
        "G1–G3 outputs do not reproduce the manuscript within tolerance. "
        "Check the exact etiLGG766 build, medium configuration, solver and software versions."
    )


In [ ]:
def top_fluxes_under_baseline(base_model, model_label: str, n: int = 20) -> pd.DataFrame:
    with base_model as work:
        set_manuscript_medium(work, 10.0, 0.0)
        work.objective = CFG.biomass_id
        work.objective_direction = "max"
        solution = work.optimize()
        if solution.status != "optimal":
            raise RuntimeError(f"{model_label} baseline solve failed: {solution.status}")
        top_ids = solution.fluxes.abs().nlargest(n).index
        rows = []
        for reaction_id in top_ids:
            reaction = work.reactions.get_by_id(reaction_id)
            rows.append(
                {
                    "model": model_label,
                    "reaction_id": reaction_id,
                    "reaction_name": reaction.name,
                    "flux": float(solution.fluxes[reaction_id]),
                    "absolute_flux": abs(float(solution.fluxes[reaction_id])),
                }
            )
        return pd.DataFrame(rows)


if not CFG.base_model_path.exists():
    raise FileNotFoundError(f"Base model required for Figure 3: {CFG.base_model_path}")
base_ilgg = cobra.io.read_sbml_model(str(CFG.base_model_path))
figure3_fluxes = pd.concat(
    [
        top_fluxes_under_baseline(base_ilgg, "iLGG766 (unconstrained)"),
        top_fluxes_under_baseline(model, "etiLGG766 (dual constrained)"),
    ],
    ignore_index=True,
)
figure3_fluxes.to_csv(CFG.output_dir / "Figure_3_Top20_Flux_Comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 7.2), sharex=False)
for ax, (label, group) in zip(axes, figure3_fluxes.groupby("model", sort=False)):
    plot_data = group.sort_values("flux")
    colors = np.where(plot_data["flux"] >= 0, "#4C78A8", "#E45756")
    ax.barh(plot_data["reaction_id"], plot_data["flux"], color=colors)
    ax.axvline(0.0, color="#333333", linewidth=0.8)
    ax.set_title(label)
    ax.set_xlabel("Flux (mmol gDW$^{-1}$ h$^{-1}$)")
axes[0].set_ylabel("Reaction")
fig.suptitle("Effect of enzyme and thermodynamic constraints on high-magnitude fluxes")
fig.tight_layout()
fig.savefig(CFG.output_dir / "Figure_3_Top20_Flux_Comparison.png", bbox_inches="tight")
plt.show()


In [ ]:
def run_bayesian_optimization(base_model):
    evaluation_records: list[dict] = []

    def objective(point: Sequence[float]) -> float:
        glucose, ascorbate = map(float, point)
        result = evaluate_condition(base_model, glucose, ascorbate, growth_fraction=CFG.growth_fraction)
        result["evaluation"] = len(evaluation_records) + 1
        evaluation_records.append(result)
        if result["status"] != "optimal" or not np.isfinite(result["acetate"]):
            return 1.0e6
        return -float(result["acetate"])

    space = [
        Real(0.0, 50.0, name="glucose_uptake"),
        Real(0.0, 30.0, name="ascorbate_uptake"),
    ]
    total_calls = CFG.bo_initial_points + CFG.bo_guided_iterations
    result = gp_minimize(
        objective,
        dimensions=space,
        n_calls=total_calls,
        n_initial_points=CFG.bo_initial_points,
        initial_point_generator="random",
        acq_func="EI",
        random_state=CFG.random_seed,
        noise=1.0e-10,
    )
    history = pd.DataFrame(evaluation_records)
    history["best_so_far_acetate"] = history["acetate"].cummax()
    if len(history) != total_calls:
        raise AssertionError(f"Expected {total_calls} BO evaluations, observed {len(history)}.")
    return result, history


bo_result, bo_history = run_bayesian_optimization(model)
bo_history.to_csv(CFG.output_dir / "Figure_4_Bayesian_Optimization_History.csv", index=False)

bo_best = bo_history.loc[bo_history["acetate"].idxmax()].copy()
bo_audit = pd.DataFrame(
    [
        {"quantity": "evaluations", "observed": len(bo_history), "expected": 38, "tolerance": 0, "pass": len(bo_history) == 38},
        {"quantity": "glucose", "observed": bo_best["glucose_bound"], "expected": 30.33, "tolerance": 1.0, "pass": abs(bo_best["glucose_bound"] - 30.33) <= 1.0},
        {"quantity": "ascorbate", "observed": bo_best["ascorbate_bound"], "expected": 30.00, "tolerance": 1.0, "pass": abs(bo_best["ascorbate_bound"] - 30.00) <= 1.0},
        {"quantity": "acetate", "observed": bo_best["acetate"], "expected": 126.54, "tolerance": MODEL_TOLERANCES["acetate"], "pass": abs(bo_best["acetate"] - 126.54) <= MODEL_TOLERANCES["acetate"]},
        {"quantity": "growth", "observed": bo_best["growth"], "expected": 0.38, "tolerance": MODEL_TOLERANCES["growth"], "pass": abs(bo_best["growth"] - 0.38) <= MODEL_TOLERANCES["growth"]},
    ]
)
bo_audit.to_csv(CFG.output_dir / "Bayesian_Optimization_Manuscript_Audit.csv", index=False)
display(bo_audit)
if CFG.strict_numeric_audit:
    assert bo_audit["pass"].all(), "Bayesian optimization did not reproduce the manuscript optimum within tolerance."

fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.scatter(bo_history["evaluation"], bo_history["acetate"], s=28, alpha=0.65, color="#4C78A8", label="Model evaluations")
ax.plot(bo_history["evaluation"], bo_history["best_so_far_acetate"], color="#D1495B", linewidth=2.2, label="Best so far")
ax.axhline(53.53, color="#666666", linestyle="--", linewidth=1.3, label="G1 baseline (audit target)")
ax.set(xlabel="Evaluation", ylabel="Maximum acetate secretion (mmol gDW$^{-1}$ h$^{-1}$)", title="Bayesian optimization convergence")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(CFG.output_dir / "Figure_4_Bayesian_Optimization_Convergence.png", bbox_inches="tight")
plt.show()


In [ ]:
def nondominated_maxima(frame: pd.DataFrame, columns: Sequence[str]) -> pd.Series:
    values = frame.loc[:, columns].to_numpy(dtype=float)
    keep = np.ones(len(frame), dtype=bool)
    for i, current in enumerate(values):
        dominates_i = np.all(values >= current, axis=1) & np.any(values > current, axis=1)
        keep[i] = not dominates_i.any()
    return pd.Series(keep, index=frame.index)


def build_pareto_landscape(base_model, design_points: pd.DataFrame) -> pd.DataFrame:
    rows = []
    fractions = np.linspace(0.50, 1.00, 11)
    unique_designs = (
        design_points[["glucose_bound", "ascorbate_bound"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    fixed_designs = MANUSCRIPT_CONDITIONS.rename(columns={"glucose": "glucose_bound", "ascorbate": "ascorbate_bound"})
    unique_designs = pd.concat([unique_designs, fixed_designs[["glucose_bound", "ascorbate_bound"]]], ignore_index=True).drop_duplicates()
    for design in unique_designs.itertuples(index=False):
        for fraction in fractions:
            rows.append(evaluate_condition(base_model, design.glucose_bound, design.ascorbate_bound, growth_fraction=float(fraction)))
    landscape = pd.DataFrame(rows)
    feasible = landscape["status"].eq("optimal") & landscape[["growth", "acetate"]].notna().all(axis=1)
    landscape["is_pareto"] = False
    landscape.loc[feasible, "is_pareto"] = nondominated_maxima(landscape.loc[feasible], ["growth", "acetate"])
    return landscape


pareto_landscape = build_pareto_landscape(model, bo_history)
pareto_landscape.to_csv(CFG.output_dir / "Figure_5_Growth_Acetate_Pareto_Landscape.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 5.0))
feasible = pareto_landscape[pareto_landscape["status"] == "optimal"]
scatter = ax.scatter(feasible["growth"], feasible["acetate"], c=feasible["ascorbate_bound"], cmap="viridis", s=24, alpha=0.55)
front = feasible[feasible["is_pareto"]].sort_values("growth")
ax.scatter(front["growth"], front["acetate"], marker="D", s=52, facecolor="#35A853", edgecolor="black", linewidth=0.4, label="Pareto optimal")
ax.scatter([bo_best["growth"]], [bo_best["acetate"]], marker="*", s=180, color="#F2C14E", edgecolor="black", linewidth=0.5, label="Bayesian optimum")
ax.set(xlabel="Growth rate (h$^{-1}$)", ylabel="Acetate secretion (mmol gDW$^{-1}$ h$^{-1}$)", title="Growth–acetate Pareto landscape")
fig.colorbar(scatter, ax=ax, label="L-ascorbate uptake bound")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(CFG.output_dir / "Figure_5_Growth_Acetate_Pareto_Front.png", bbox_inches="tight")
plt.show()


In [ ]:
def counterfactual_capacity(
    base_model,
    *,
    reaction_knockouts: Sequence[str] = (),
    gene_knockouts: Sequence[str] = (),
) -> dict:
    no_asc = evaluate_condition(
        base_model, 30.33, 0.0, growth_fraction=0.90,
        reaction_knockouts=reaction_knockouts, gene_knockouts=gene_knockouts,
    )
    forced_asc = evaluate_condition(
        base_model, 30.33, 30.0, growth_fraction=0.90, force_ascorbate=True,
        reaction_knockouts=reaction_knockouts, gene_knockouts=gene_knockouts,
    )
    both_feasible = no_asc["status"] == "optimal" and forced_asc["status"] == "optimal"
    delta = forced_asc["acetate"] - no_asc["acetate"] if both_feasible else np.nan
    return {
        "noAsc_status": no_asc["status"],
        "noAsc_mu_max": no_asc["mu_max"],
        "noAsc_growth_floor": no_asc["growth_floor"],
        "noAsc_growth": no_asc["growth"],
        "noAsc_acetate_max": no_asc["acetate"],
        "forcedAsc_status": forced_asc["status"],
        "forcedAsc_mu_max": forced_asc["mu_max"],
        "forcedAsc_growth_floor": forced_asc["growth_floor"],
        "forcedAsc_growth": forced_asc["growth"],
        "forcedAsc_actual_uptake": forced_asc["ascorbate_uptake"],
        "forcedAsc_acetate_max": forced_asc["acetate"],
        "delta_acetate_capacity": delta,
    }


wt_counterfactual = counterfactual_capacity(model)
counterfactual_df = pd.DataFrame([wt_counterfactual])
counterfactual_df.to_csv(CFG.output_dir / "Figure_6_Counterfactual_DeltaAc.csv", index=False)
display(counterfactual_df)

if wt_counterfactual["forcedAsc_status"] == "optimal":
    assert np.isclose(wt_counterfactual["forcedAsc_actual_uptake"], 30.0, atol=1e-6), "forcedAsc did not enforce uptake = 30."

fig, ax = plt.subplots(figsize=(5.8, 4.4))
no_capacity = wt_counterfactual["noAsc_acetate_max"]
delta_capacity = wt_counterfactual["delta_acetate_capacity"]
ax.bar(["noAsc", "forcedAsc"], [no_capacity, no_capacity], color="#4C78A8", label="noAsc maximum capacity")
ax.bar(["noAsc", "forcedAsc"], [0.0, delta_capacity], bottom=[no_capacity, no_capacity], color="#F28E2B", label="additional capacity (ΔAc)")
ax.set(ylabel="Maximum acetate secretion (mmol gDW$^{-1}$ h$^{-1}$)", title="G2-background counterfactual acetate capacity")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(CFG.output_dir / "Figure_6_Counterfactual_DeltaAc.png", bbox_inches="tight")
plt.show()


In [ ]:
def flux_envelope(base_model, uptake_grid: Iterable[float] = np.linspace(0.0, 30.0, 61)) -> pd.DataFrame:
    rows = []
    for uptake in map(float, uptake_grid):
        lower = evaluate_condition(base_model, 30.33, uptake, growth_fraction=0.90, force_ascorbate=True, acetate_sense="min")
        upper = evaluate_condition(base_model, 30.33, uptake, growth_fraction=0.90, force_ascorbate=True, acetate_sense="max")
        rows.append(
            {
                "ascorbate_uptake": uptake,
                "lower_status": lower["status"],
                "upper_status": upper["status"],
                "mu_max": upper["mu_max"],
                "growth_floor": upper["growth_floor"],
                "acetate_min": lower["acetate"] if lower["status"] == "optimal" else np.nan,
                "acetate_max": upper["acetate"] if upper["status"] == "optimal" else np.nan,
            }
        )
    return pd.DataFrame(rows)


envelope_df = flux_envelope(model)
envelope_df.to_csv(CFG.output_dir / "Ascorbate_Acetate_Flux_Envelope.csv", index=False)
display(envelope_df.head())

fig, ax = plt.subplots(figsize=(6.6, 4.6))
feasible = envelope_df.dropna(subset=["acetate_min", "acetate_max"])
ax.fill_between(feasible["ascorbate_uptake"], feasible["acetate_min"], feasible["acetate_max"], color="#72B7B2", alpha=0.28)
ax.plot(feasible["ascorbate_uptake"], feasible["acetate_min"], color="#4C78A8", label="Minimum acetate")
ax.plot(feasible["ascorbate_uptake"], feasible["acetate_max"], color="#E45756", label="Maximum acetate")
ax.set(xlabel="Fixed L-ascorbate uptake (mmol gDW$^{-1}$ h$^{-1}$)", ylabel="Acetate secretion (mmol gDW$^{-1}$ h$^{-1}$)", title="L-ascorbate–acetate flux envelope")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(CFG.output_dir / "Ascorbate_Acetate_Flux_Envelope.png", bbox_inches="tight")
plt.show()


In [ ]:
DEFAULT_ASCORBATE_TARGETS = [
    "EX_ascb__L_e", "ASCBpts", "ASCBPL", "KG6PDC", "X5PL3E", "RBP4E",
    "PKETX", "TKT1_REV_EXP_1", "PRPPS", "GLUPRT", "PPRGL", "FGFT_1",
    "GHMT2r", "MTHFR3", "METS", "METGL_EXP_1", "AHSERL", "ACt2r_1", "EX_ac_e",
]


def load_ascorbate_targets(base_model) -> list[str]:
    if CFG.target_table_path.exists():
        targets = pd.read_csv(CFG.target_table_path)
        if "reaction_id" not in targets.columns:
            raise ValueError(f"{CFG.target_table_path} must contain a reaction_id column.")
        reaction_ids = targets["reaction_id"].dropna().astype(str).tolist()
    else:
        warnings.warn("Curated target table is absent; using the documented default PTS/ula target list.")
        reaction_ids = DEFAULT_ASCORBATE_TARGETS
    missing = [rid for rid in reaction_ids if rid not in base_model.reactions]
    if missing:
        warnings.warn(f"Target reactions absent from this model and skipped: {missing}")
    return [rid for rid in reaction_ids if rid in base_model.reactions]


def maximum_ascorbate_uptake(
    base_model,
    *,
    reaction_knockouts: Sequence[str] = (),
    gene_knockouts: Sequence[str] = (),
) -> dict:
    with base_model as work:
        set_manuscript_medium(work, 30.33, 30.0, force_ascorbate=False)
        for reaction_id in reaction_knockouts:
            _reaction(work, reaction_id).knock_out()
        if gene_knockouts:
            knock_out_model_genes(work, list(gene_knockouts))

        biomass = _reaction(work, CFG.biomass_id)
        work.objective = biomass
        work.objective_direction = "max"
        growth_solution = work.optimize()
        if growth_solution.status != "optimal":
            return {"uptake_status": f"growth:{growth_solution.status}", "max_ascorbate_uptake": np.nan}
        mu_max = float(growth_solution.objective_value)
        biomass.lower_bound = max(float(biomass.lower_bound), 0.90 * mu_max)
        asc = _reaction(work, CFG.ascorbate_exchange_id)
        work.objective = asc
        work.objective_direction = "min"
        solution = work.optimize()
        uptake = max(0.0, -float(solution.fluxes[CFG.ascorbate_exchange_id])) if solution.status == "optimal" else np.nan
        return {"uptake_status": solution.status, "max_ascorbate_uptake": uptake}


def classify_knockout(max_uptake: float, cf: dict, wt_delta: float, eps: float = 1e-6) -> dict:
    uptake_essential = (not np.isfinite(max_uptake)) or max_uptake <= eps
    utilization_essential = cf["forcedAsc_status"] != "optimal" and cf["noAsc_status"] == "optimal"
    knockout_delta = cf["delta_acetate_capacity"]
    delta_essential = bool(
        np.isfinite(wt_delta) and wt_delta > eps and (
            utilization_essential or (np.isfinite(knockout_delta) and knockout_delta <= eps)
        )
    )
    labels = [
        label for flag, label in [
            (uptake_essential, "uptake-essential"),
            (utilization_essential, "utilization-essential"),
            (delta_essential, "DeltaAc-essential"),
        ] if flag
    ]
    return {
        "uptake_essential": uptake_essential,
        "utilization_essential": utilization_essential,
        "deltaAc_essential": delta_essential,
        "classification": ";".join(labels) if labels else "not-essential-under-tested-conditions",
    }


def knockout_evidence_table(base_model) -> pd.DataFrame:
    target_ids = load_ascorbate_targets(base_model)
    wt_delta = counterfactual_capacity(base_model)["delta_acetate_capacity"]
    rows = []

    for reaction_id in target_ids:
        reaction = base_model.reactions.get_by_id(reaction_id)
        uptake = maximum_ascorbate_uptake(base_model, reaction_knockouts=[reaction_id])
        cf = counterfactual_capacity(base_model, reaction_knockouts=[reaction_id])
        row = {
            "target_type": "reaction", "target_id": reaction_id, "target_name": reaction.name,
            "reaction_ids": reaction_id, "reaction_names": reaction.name,
            "model_gene_protein_ids": ";".join(sorted(g.id for g in reaction.genes)),
            "gpr": reaction.gene_reaction_rule,
            **uptake, **cf,
            **classify_knockout(uptake["max_ascorbate_uptake"], cf, wt_delta),
        }
        rows.append(row)

    gene_ids = sorted({gene.id for rid in target_ids for gene in base_model.reactions.get_by_id(rid).genes})
    for gene_id in gene_ids:
        gene = base_model.genes.get_by_id(gene_id)
        associated = sorted(rxn.id for rxn in gene.reactions if rxn.id in target_ids)
        uptake = maximum_ascorbate_uptake(base_model, gene_knockouts=[gene_id])
        cf = counterfactual_capacity(base_model, gene_knockouts=[gene_id])
        row = {
            "target_type": "gene", "target_id": gene_id, "target_name": gene.name,
            "reaction_ids": ";".join(associated),
            "reaction_names": ";".join(base_model.reactions.get_by_id(rid).name for rid in associated),
            "model_gene_protein_ids": gene_id,
            "gpr": ";".join(base_model.reactions.get_by_id(rid).gene_reaction_rule for rid in associated),
            **uptake, **cf,
            **classify_knockout(uptake["max_ascorbate_uptake"], cf, wt_delta),
        }
        rows.append(row)
    return pd.DataFrame(rows)


knockout_df = knockout_evidence_table(model)
knockout_path = CFG.output_dir / "Supplementary_Table_S1_Ascorbate_Essential_Node_Evidence.csv"
knockout_df.to_csv(knockout_path, index=False)
display(knockout_df.sort_values(["classification", "target_type", "target_id"]))


In [ ]:
PERTURBATION_CODES = {"unchanged": 0, "inactivated": 1, "forward_only": 2, "reverse_only": 3}


def allowed_perturbations(reaction) -> list[str]:
    operations = ["inactivated"]
    if reaction.lower_bound < 0 < reaction.upper_bound:
        operations.append("forward_only")
        operations.append("reverse_only")
    return operations


def apply_structural_perturbation(reaction, operation: str) -> None:
    lower, upper = map(float, reaction.bounds)
    if operation == "inactivated":
        reaction.bounds = (0.0, 0.0)
    elif operation == "forward_only":
        reaction.bounds = (max(0.0, lower), max(0.0, upper))
    elif operation == "reverse_only":
        reaction.bounds = (min(0.0, lower), min(0.0, upper))
    else:
        raise ValueError(operation)


def generate_structural_ensemble(
    base_model,
    core_reaction_ids: set[str],
    *,
    target_viable: int = 80,
    max_attempts: int = 600,
    random_seed: int = CFG.random_seed,
    strict_manuscript_counts: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(random_seed)
    candidate_ids = [
        rxn.id for rxn in base_model.reactions
        if rxn.id not in core_reaction_ids and rxn not in base_model.exchanges and (rxn.lower_bound < 0 or rxn.upper_bound > 0)
    ]
    if len(candidate_ids) < 18:
        raise ValueError("At least 18 perturbable non-core reactions are required.")

    feature_rows, audit_rows = [], []
    viable = 0
    attempts = 0
    while viable < target_viable and attempts < max_attempts:
        attempts += 1
        n_changes = int(rng.integers(3, 19))
        selected = rng.choice(candidate_ids, size=n_changes, replace=False).tolist()
        state = {rid: PERTURBATION_CODES["unchanged"] for rid in candidate_ids}

        with base_model as work:
            set_manuscript_medium(work, 10.0, 0.0)
            for reaction_id in selected:
                reaction = work.reactions.get_by_id(reaction_id)
                operation = str(rng.choice(allowed_perturbations(reaction)))
                apply_structural_perturbation(reaction, operation)
                state[reaction_id] = PERTURBATION_CODES[operation]
                audit_rows.append({"attempt": attempts, "reaction_id": reaction_id, "operation": operation})
            work.objective = CFG.biomass_id
            work.objective_direction = "max"
            solution = work.optimize()

        biomass = float(solution.objective_value) if solution.status == "optimal" else np.nan
        accepted = solution.status == "optimal" and np.isfinite(biomass) and biomass > 1e-6
        for row in audit_rows:
            if row["attempt"] == attempts:
                row.update({"solver_status": solution.status, "biomass": biomass, "accepted": accepted})
        if accepted:
            viable += 1
            state.update({"member_id": f"member_{viable:03d}", "attempt": attempts, "biomass": biomass})
            feature_rows.append(state)

    if viable != target_viable:
        raise RuntimeError(f"Only {viable}/{target_viable} viable variants were obtained after {attempts} attempts.")
    discarded = attempts - viable
    if strict_manuscript_counts and (attempts != 111 or discarded != 31):
        raise AssertionError(
            f"Ensemble count mismatch: manuscript expects 111 attempts and 31 discarded; observed {attempts} and {discarded}."
        )
    return pd.DataFrame(feature_rows), pd.DataFrame(audit_rows)


def fit_ensemble_random_forest(feature_df: pd.DataFrame):
    meta_columns = {"member_id", "attempt", "biomass"}
    feature_columns = [column for column in feature_df.columns if column not in meta_columns]
    ordered = feature_df["biomass"].sort_values(kind="mergesort").index.to_numpy()
    labels = pd.Series(0, index=feature_df.index, dtype=int)
    labels.loc[ordered[len(ordered) // 2:]] = 1
    X = feature_df[feature_columns].astype(np.int8)
    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=0.30, stratify=labels, random_state=CFG.random_seed,
    )
    classifier = RandomForestClassifier(
        n_estimators=500, criterion="gini", class_weight="balanced",
        random_state=CFG.random_seed, n_jobs=-1,
    )
    classifier.fit(X_train, y_train)
    importance = pd.DataFrame(
        {"reaction_id": feature_columns, "importance": classifier.feature_importances_}
    ).sort_values("importance", ascending=False)
    metrics = {
        "train_size": len(X_train), "test_size": len(X_test),
        "test_accuracy": float(classifier.score(X_test, y_test)),
        "n_estimators": classifier.n_estimators, "criterion": classifier.criterion,
        "class_weight": classifier.class_weight, "random_seed": CFG.random_seed,
    }
    return classifier, importance, metrics


if not CFG.core_table_path.exists():
    raise FileNotFoundError(f"Manually curated core-reaction table is required: {CFG.core_table_path}")
core_table = pd.read_csv(CFG.core_table_path)
if "reaction_id" not in core_table.columns:
    raise ValueError("core_reactions.csv must contain a reaction_id column.")
core_ids = set(core_table["reaction_id"].dropna().astype(str))

ensemble_features, ensemble_audit = generate_structural_ensemble(base_ilgg, core_ids)
rf_model, reaction_importance, rf_metrics = fit_ensemble_random_forest(ensemble_features)
reaction_importance["reaction_name"] = reaction_importance["reaction_id"].map(
    {rxn.id: rxn.name for rxn in base_ilgg.reactions}
)
reaction_importance["gpr"] = reaction_importance["reaction_id"].map(
    {rxn.id: rxn.gene_reaction_rule for rxn in base_ilgg.reactions}
)
ensemble_features.to_csv(CFG.output_dir / "ensemble_structural_features.csv", index=False)
ensemble_audit.to_csv(CFG.output_dir / "ensemble_perturbation_audit.csv", index=False)
reaction_importance.to_csv(CFG.output_dir / "AMMEDEUS_reaction_importance.csv", index=False)
(CFG.output_dir / "AMMEDEUS_random_forest_metrics.json").write_text(
    json.dumps(rf_metrics, indent=2), encoding="utf-8"
)

top20 = reaction_importance.head(20).sort_values("importance")
fig, ax = plt.subplots(figsize=(7.2, 6.4))
ax.barh(top20["reaction_id"], top20["importance"], color="#4C78A8")
ax.set(xlabel="Random-forest feature importance", ylabel="Reaction", title="AMMEDEUS-style structural feature ranking")
fig.tight_layout()
fig.savefig(CFG.output_dir / "Figure_2_AMMEDEUS_Top20.png", bbox_inches="tight")
plt.show()


In [ ]:
prediction_error = pd.DataFrame(
    {
        "condition": ["G1", "G2", "G3"],
        "growth_predicted": [0.49, 0.38, 0.44],
        "growth_experimental_mean": [0.46, 0.34, 0.41],
        "acetate_predicted": [53.53, 126.54, 121.76],
        "acetate_experimental_mean": [43.3, 96.6, 89.9],
    }
)


def absolute_percentage_error(predicted: pd.Series, observed: pd.Series) -> pd.Series:
    if (observed == 0).any():
        raise ZeroDivisionError("APE is undefined for an observed value of zero.")
    return (predicted - observed).abs() / observed.abs() * 100.0


prediction_error["growth_APE_percent"] = absolute_percentage_error(
    prediction_error["growth_predicted"], prediction_error["growth_experimental_mean"]
)
prediction_error["acetate_APE_percent"] = absolute_percentage_error(
    prediction_error["acetate_predicted"], prediction_error["acetate_experimental_mean"]
)
growth_mape = float(prediction_error["growth_APE_percent"].mean())
acetate_mape = float(prediction_error["acetate_APE_percent"].mean())
prediction_error.to_csv(CFG.output_dir / "Table_4_APE_MAPE.csv", index=False)
display(prediction_error)
print(f"Growth MAPE: {growth_mape:.2f}%")
print(f"Acetate MAPE: {acetate_mape:.2f}%")
assert round(growth_mape, 2) == 8.53
assert round(acetate_mape, 2) == 30.02


In [ ]:
required_outputs = [
    CFG.output_dir / "Table_1_Model_Predicted_Phenotypes.csv",
    CFG.output_dir / "Figure_3_Top20_Flux_Comparison.csv",
    CFG.output_dir / "Figure_4_Bayesian_Optimization_History.csv",
    CFG.output_dir / "Figure_5_Growth_Acetate_Pareto_Landscape.csv",
    CFG.output_dir / "Figure_6_Counterfactual_DeltaAc.csv",
    CFG.output_dir / "Ascorbate_Acetate_Flux_Envelope.csv",
    CFG.output_dir / "Supplementary_Table_S1_Ascorbate_Essential_Node_Evidence.csv",
    CFG.output_dir / "ensemble_structural_features.csv",
    CFG.output_dir / "ensemble_perturbation_audit.csv",
    CFG.output_dir / "AMMEDEUS_reaction_importance.csv",
    CFG.output_dir / "Table_4_APE_MAPE.csv",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(f"The workflow is incomplete; missing outputs: {missing_outputs}")

output_records = [
    {"path": str(path), "size_bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for path in sorted(CFG.output_dir.glob("*")) if path.is_file()
]
manifest = {
    "environment": environment,
    "authoritative_model": str(CFG.pytfa_model_path),
    "authoritative_model_sha256": sha256_file(CFG.pytfa_model_path),
    "parameters": {
        "growth_fraction": CFG.growth_fraction,
        "bo_initial_points": CFG.bo_initial_points,
        "bo_guided_iterations": CFG.bo_guided_iterations,
        "random_seed": CFG.random_seed,
        "G2_glucose": 30.33,
        "G2_forced_ascorbate": 30.00,
    },
    "outputs": output_records,
}
manifest_path = CFG.output_dir / "reproducibility_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Workflow complete. Manifest: {manifest_path}")
